# RG3 gas: current-shot innovation confirmation of extra expert detections
Frozen original RF + expert extra branch. Six signed innovation hypotheses; gate feature/quantile/expert cutoff chosen only from training inner predictions.
Motivated by prior observed missed case and false alarms: exploratory test-informed ideation, not untouched independent validation.
Current-shots measured delta/departure are causal; no future sensor. Normal-only quantile fitting, explicit outer-block/timestamp exclusion assertions.
Two selectors: unconstrained inner-F1 confirmation (deterministic physics-priority tie-break), and evidence-supported selector requiring extra inner TP. Report both; tie-break discoveries can be weak evidence.
No automatic promotion/commit/push.

In [ ]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.metrics import confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
outer=pd.read_csv(tables/'rg3_rank_fusion_predictions.csv',parse_dates=['TimeStamp'])
outer=outer.loc[outer.model.eq('baseline_rank')].copy()
inside=pd.read_csv(tables/'rg3_rank_fusion_inner_scores.csv',parse_dates=['TimeStamp'])
d=pd.read_csv(tables/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
d=d.loc[d['product'].eq('RG3')].copy()
fields=['Barrel_Temperature_5__delta1','Barrel_Temperature_5__dev10','Max_Injection_Pressure__delta1','Max_Injection_Pressure__dev10','Cushion_Position__delta1']
assert not d.TimeStamp.duplicated().any()
outer=outer.merge(d[['TimeStamp']+fields],on='TimeStamp',validate='many_to_one')
inside=inside.merge(d[['TimeStamp']+fields],on='TimeStamp',validate='many_to_one')
original=pd.read_csv(tables/'rg3_nested_ensemble_predictions.csv',parse_dates=['TimeStamp'])
original=original.loc[original.model.eq('max_union'),['scheme','TimeStamp','predicted_train_fpr1']]
def stats(yy,p):
    tn,fp,fn,tp=confusion_matrix(yy,p,labels=[0,1]).ravel()
    return int(tp),int(fp),int(fn),int(tn),2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.
def features(cal,q):
    def raw(z):
        return {'barrel5_rise':z.Barrel_Temperature_5__delta1.to_numpy(float),'barrel5_hot_departure':z.Barrel_Temperature_5__dev10.to_numpy(float),'pressure_rise':z.Max_Injection_Pressure__delta1.to_numpy(float),'pressure_high_departure':z.Max_Injection_Pressure__dev10.to_numpy(float),'cushion_loss':-z.Cushion_Position__delta1.to_numpy(float)}
    a=raw(cal);b=raw(q)
    # Both must rise; scales come only from internal train normals.
    norm=cal.actual.eq(0).to_numpy()
    scales=[]
    for name in ['barrel5_rise','pressure_rise']:
        x=a[name][norm&np.isfinite(a[name])]
        scale=max(float(np.quantile(x,.75)-np.quantile(x,.25)),.1) if len(x) else .1
        scales.append(scale)
    a['thermal_pressure_joint']=np.minimum(a['barrel5_rise']/scales[0],a['pressure_rise']/scales[1])
    b['thermal_pressure_joint']=np.minimum(b['barrel5_rise']/scales[0],b['pressure_rise']/scales[1])
    return a,b
records=[];choices=[];candidate_audit=[];foldaudit=[]
rates=[.0025,.005,.01];quantiles=[.90,.95,.975]
for (scheme,block),q in outer.groupby(['scheme','block'],sort=False):
    cal=inside.loc[inside.scheme.eq(scheme)&inside.heldout_block.eq(block)].dropna(subset=['raw_baseline_score','raw_expert_score']).reset_index(drop=True)
    assert not cal.block.eq(block).any()
    assert set(cal.TimeStamp).isdisjoint(set(q.TimeStamp))
    normal=cal.actual.eq(0).to_numpy()
    cut_b=float(np.quantile(cal.loc[normal,'raw_baseline_score'],.99,method='higher'))
    base_inner=cal.raw_baseline_score.gt(cut_b).to_numpy()
    base_outer=q.raw_baseline_score.gt(cut_b).to_numpy()
    tp0,fp0,fn0,tn0,f0=stats(cal.actual,base_inner)
    a,b=features(cal,q);candidates=[]
    for name,v in a.items():
        valid=normal&np.isfinite(v)
        for quantile in quantiles:
            cut_f=float(np.quantile(v[valid],quantile,method='higher')) if valid.any() else np.inf
            for rate in rates:
                cut_e=float(np.quantile(cal.loc[normal,'raw_expert_score'],1-rate,method='higher'))
                inner_add=cal.raw_expert_score.gt(cut_e).to_numpy()&(v>cut_f)
                out_add=q.raw_expert_score.gt(cut_e).to_numpy()&(b[name]>cut_f)
                pr=base_inner|inner_add;out=base_outer|out_add
                tp,fp,fn,tn,f=stats(cal.actual,pr)
                model=f'{name}_q{quantile}_expert{rate}'
                c=dict(model=model,feature=name,quantile=quantile,expert_budget=rate,baseline_threshold=cut_b,expert_threshold=cut_e,feature_threshold=cut_f,inner_F1=f,inner_TP=tp,inner_FP=fp,inner_added_TP=tp-tp0,inner_added_FP=fp-fp0)
                candidates.append((c,out))
                candidate_audit.append(dict(scheme=scheme,block=block,**c))
                for j,r in enumerate(q.itertuples()):records.append(dict(scheme=scheme,model=model,TimeStamp=r.TimeStamp,block=block,actual=int(r.actual),prediction=int(out[j]),baseline_prediction=int(base_outer[j])))
    # Priority is explicitly fixed before observing this experiment's outer metrics.
    def priority(cp):
        c=cp[0]
        return (c['inner_F1'],-c['inner_FP'],c['feature']=='barrel5_rise',-abs(c['quantile']-.95),-abs(c['expert_budget']-.005))
    selected,out=max(candidates,key=priority)
    selected_rows=[('inner_confirmation_selected',selected,out)]
    guarded=[cp for cp in candidates if cp[0]['inner_added_TP']>=1 and cp[0]['inner_added_FP']<=2 and cp[0]['inner_F1']>f0]
    if guarded:c,p=max(guarded,key=priority)
    else:
        c=dict(model='baseline_only',feature='none',quantile=np.nan,expert_budget=np.nan,baseline_threshold=cut_b,expert_threshold=np.nan,feature_threshold=np.nan,inner_F1=f0,inner_TP=tp0,inner_FP=fp0,inner_added_TP=0,inner_added_FP=0)
        p=base_outer
    selected_rows.append(('inner_supported_selection',c,p))
    for label,c,out in selected_rows:
        choices.append(dict(scheme=scheme,block=block,selector=label,**c))
        for j,r in enumerate(q.itertuples()):records.append(dict(scheme=scheme,model=label,TimeStamp=r.TimeStamp,block=block,actual=int(r.actual),prediction=int(out[j]),baseline_prediction=int(base_outer[j])))
    foldaudit.append(dict(scheme=scheme,block=block,inner_rows=len(cal),inner_gas=int(cal.actual.sum()),test_rows=len(q),test_gas=int(q.actual.sum()),timestamp_overlap=0,heldout_block_in_cal=False))
pp=pd.DataFrame(records);rows=[]
for (scheme,name),g in pp.groupby(['scheme','model'],sort=False):
    tp,fp,fn,tn,f=stats(g.actual,g.prediction)
    rows.append(dict(scheme=scheme,model=name,TP=tp,FP=fp,FN=fn,TN=tn,F1=f))
metrics=pd.DataFrame(rows)
pp.to_csv(tables/'rg3_innovation_confirmation_predictions.csv',index=False,encoding='utf-8-sig')
metrics.to_csv(tables/'rg3_innovation_confirmation_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(choices).to_csv(tables/'rg3_innovation_confirmation_inner_choices.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(candidate_audit).to_csv(tables/'rg3_innovation_confirmation_inner_candidates.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(foldaudit).to_csv(tables/'rg3_innovation_confirmation_fold_audit.csv',index=False,encoding='utf-8-sig')
check=pp.loc[pp.model.eq('inner_confirmation_selected')].merge(original,on=['scheme','TimeStamp'],validate='one_to_one')
assert len(check)==1023 and (check.baseline_prediction.astype(bool)==check.predicted_train_fpr1).all()
assert (check.prediction>=check.baseline_prediction).all()
print(metrics.loc[metrics.model.str.startswith('inner_')].to_string(index=False))
print(pd.DataFrame(choices).to_string(index=False))
print(metrics.loc[metrics.scheme.eq('leave_one_block_out')].sort_values(['F1','FP'],ascending=[False,True]).head(10).to_string(index=False))


## Executed results
Inner-confirmation-selected whole TP5 FP6 FN12 TN560 F1=0.357; forward TP3 FP4 FN10 TN423 F1=0.300. Supported-selector fallback stays original4/6 and2/4. Internal added TP is zero; tie-priority determines feature. Exploratory, not independently validated.
See reports/rg3_current_innovation_confirmation.md and reports/rg3_remaining12_cold_signature_review.md.